In [ ]:
from glob import glob
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
import numpy as np
import os
import pandas as pd 
import seaborn as sns
from statsmodels.stats.multitest import multipletests
import sys

plt.rcParams['svg.fonttype'] = 'none'

sys.path.append("../src")
from evaluation_metrics import *
from evaluation_utils import *

In [ ]:
results = "../evaluation_results/1_4_edist/*.csv"
results_edist = "../evaluation_results/1_4_edist/with_subsampling_10k/*.csv"

In [ ]:
csvs = glob(results) + glob(results_edist)

In [ ]:
datasets = [f"mcfarland_{i}" for i in range(1, 6)] + ["bhattacherjee_Astro", "bhattacherjee_Endo", "bhattacherjee_Excitatory", "norman", "schiebinger"]

In [ ]:
dfs = list()
runtime_memory = list()
for dataset in datasets:
    dataset_df_dict = {os.path.basename(csv).split("_")[0]: pd.read_csv(csv, index_col="testgroup") for csv in csvs if dataset in csv}
    for method in dataset_df_dict:
        dataset_df_dict[method].columns = [f"{method}_{c}" for c in dataset_df_dict[method].columns]
    dataset_df = pd.concat(list(dataset_df_dict.values()), axis=1)
    dataset_df["xm_perturbation_strength"] /= np.max(dataset_df["xm_perturbation_strength"])
    dataset_df["dataset"] = dataset
    dataset_df["xm_P_adj"] = multipletests(dataset_df["xm_P"], method="fdr_bh")[1]
    dfs.append(dataset_df)
result = pd.concat(dfs)

In [ ]:
dataset_df_dict.keys()

In [ ]:
result["log1p(xm_P_adj)"] = np.log1p(result["xm_P_adj"])
result["log1p(edist_P_adj)"] = np.log1p(result["edist_P_adj"])

In [ ]:
pal = sns.color_palette("YlOrBr", as_cmap=True)

In [ ]:
f, axs = plt.subplots(2, 5, figsize=(10, 5)) #, sharey=True)
for i, (dataset, dataset_df) in enumerate(result.reset_index().groupby("dataset")):
    ax = axs[i // 5, i % 5]
    sns.scatterplot(ax=ax, data=dataset_df, x="edist_P", y="xm_P", hue="xm_perturbation_strength", marker="o", legend=False)

    splits = dataset.split("_")
    if len(splits) > 1:
        title = f"{splits[0][:3]}_{splits[1][:2]}"
    else:
        title = splits[0][:3]
    ax.set_title(title)
    ax.ticklabel_format(useMathText=True)
    
    y_vals = ax.collections[0].get_offsets()[:, 1]
    ax.set_yticks(y_vals)
    ax.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    ax.yaxis.get_major_formatter().set_useOffset(False)
    ax.yaxis.get_major_formatter().set_scientific(True)
    ax.yaxis.get_major_formatter().set_powerlimits((0, 0))
    ax.yaxis.get_major_formatter().set_useMathText(False)
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.1e'))

    ax.set_xticks([np.mean(ax.get_xticks())])
    ax.set_xlabel(None)

plt.tight_layout()
plt.savefig("../plots/fig3/fig3_edist_P-values.svg", bbox_inches="tight")

In [ ]:
(result.loc[1, "edist_time_edist_test"] + result.loc[1, "edist_time_pca"]) / (result.loc[1, "xm_time_test"] + result.loc[2, "xm_time_test"])

In [ ]:
(result.loc[1, "xm_time_test"] + result.loc[2, "xm_time_test"])

In [ ]:
val = np.mean(ax.get_yticks())
#val *= 10**(-np.log10(val))
#val

In [ ]:
np.log10(val)

In [ ]:
pal = {f"mcfarland_{i}": sns.color_palette("Blues", 5)[i-1] for i in range(1, 6)}
pal.update({"bhattacherjee_Astro": sns.color_palette("Greens", 3)[0], "bhattacherjee_Endo": sns.color_palette("Reds", 3)[1], "bhattacherjee_Excitatory": sns.color_palette("Reds", 3)[2],})
pal.update({"norman": sns.color_palette("Purples", 1)[0]})
pal.update({"schiebinger": sns.color_palette("Greys", 1)[0]})

In [ ]:
plt.figure(figsize=(2,2))
sns.scatterplot(result.sort_values("dataset", ascending=False), x="edist_relative_support", y="xm_relative_support", s=50, alpha=1, legend=False, marker="x", color="black")
plt.plot([0, 1], [0, 1], ls="--", color="grey")
plt.axis("square")
plt.savefig("../plots/fig3/fig3_edist_support.svg", bbox_inches="tight")

In [ ]:
ram_edist = result[["dataset", "edist_total_RAM"]].drop_duplicates().reset_index(drop=True).set_index("dataset")
ram_xm = result[["dataset", "xm_total_RAM"]].drop_duplicates().reset_index(drop=True).set_index("dataset")
ram = pd.concat([ram_edist, ram_xm], axis=1).reset_index(names="dataset")

In [ ]:
plt.figure(figsize=(2,2))
sns.scatterplot(ram, x="edist_total_RAM", y="xm_total_RAM", s=100, legend=False, palette=pal, marker="x", color="black")
plt.plot([0, 10], [0, 10], ls="--", color="grey")
plt.axis("square")
plt.savefig("../plots/fig3/fig3_edist_RAM.svg", bbox_inches="tight")

In [ ]:
xm_time = dict()
edist_time = dict()
for (dataset, df) in result.groupby("dataset"):
    xm_time[dataset] = df["xm_time_test"].sum()
    edist_time[dataset] = df["edist_time_pca"].iloc[0] + df["edist_time_edist_test"].iloc[0]

In [ ]:
times = pd.DataFrame([xm_time, edist_time], index=["scXMatch", "E-distance"]).T.reset_index(names="dataset")

In [ ]:
times

In [ ]:
plt.figure(figsize=(2,2))
sns.scatterplot(times, x="E-distance", y="scXMatch", s=100, legend=False, marker="x", color="black")
max_val = np.max(times[["scXMatch", "E-distance"]].values)
plt.plot([0, max_val], [0, max_val], ls="--", color="grey")
plt.yscale("symlog")
plt.xscale("symlog")
plt.axis("square")
plt.savefig("../plots/fig3/fig3_edist_time.svg", bbox_inches="tight")